In [ ]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

In [ ]:
from tabseq_torch.dl.tb_dataset import TbDataset as TbDataset

In [ ]:
from tabseq_torch.dl.padded_batch import GroupData

In [ ]:
from tabseq_torch.dl.lazy_dataset import LazyDataset

In [ ]:
from pathlib import Path
import polars as pl
from polars.testing import assert_frame_equal

import torch
import lightning as L
from lightning.pytorch.loggers import TensorBoardLogger
import torch.utils.data as data

from tabseq_torch.dl.encoder import (
    make_cat2code,
    DataCatalog,
    Cat2Code,
    Encoder,
    filter_part,
    cat2code_size,
    make_catnum,
    CatNum,
    write_json,
)
from tabseq_torch.dl.lazy_dataset import (
    LazyDataset,
)
from tabseq_torch.dl.group_data import GroupData

from tabseq_torch.dl import (
    collate_padded_batch_fn,
    dummy_collate_fn,
    random_split,
    ColumnType,
)

from tabseq_torch.lit.lstm import LitMulticlassTabLSTM

In [ ]:
pl.__version__ == "1.37.1"

# Names calasses data profiling

In [ ]:
data_path = Path("/home/jovyan/data/tbts_data")

In [ ]:
data_path.exists()

In [ ]:
for p in data_path.iterdir():
    print(p)

In [ ]:
src_path = data_path / "tbts-names.parquet"

In [ ]:
pl.scan_parquet(src_path).collect_schema()

In [ ]:
for p in Path(data_path / "encoded").iterdir():
    print(p)

In [ ]:
targets_path = data_path / "encoded" / "labels.parquet"
pl.scan_parquet(targets_path).head(5).collect()

# Init the data catalog

In [ ]:
dc = DataCatalog(
    src_tabts_path=src_path,
    cat_path=src_path.parent / "cat",
    meta_path=src_path.parent / "meta",
)

# Encoding and metadata

In [ ]:
encodeed_cat_columns = ["name_chars", "vc"]
encodeed_cat_columns_setname = "allcat"

In [ ]:
cat2code: Cat2Code = make_cat2code(src_path, encodeed_cat_columns)

In [ ]:
for k in cat2code.keys():
    print(k)

In [ ]:
dc.cat2code_json(encodeed_cat_columns_setname)

In [ ]:
write_json(data=cat2code, path=dc.cat2code_json(encodeed_cat_columns_setname))

In [ ]:
catnum: CatNum = make_catnum(cat2code)

In [ ]:
catnum

In [ ]:
write_json(data=catnum, path=dc.catnum_json(encodeed_cat_columns_setname))

# Integration tests

In [ ]:
lazy_dataset = LazyDataset(
    cat_path=data_path / "encoded" / "cat.parquet",
    catnum_json=dc.catnum_json(encodeed_cat_columns_setname),
    cat_columns_to_select=encodeed_cat_columns,
    groupby_column="G",
    num_chunks=1,
)

In [ ]:
len(lazy_dataset)

In [ ]:
lazy_dataset.set_targetmeta(data_path=targets_path, target_column="labels")

In [ ]:
gd: GroupData = next(iter(lazy_dataset))

In [ ]:
print(gd)

In [ ]:
loader = data.DataLoader(
    dataset=lazy_dataset,
    batch_size=4,
    collate_fn=dummy_collate_fn,
    num_workers=0,
)

In [ ]:
n = 0
for batch in loader:
    for d in batch:
        n += len(d)
print(n)

In [ ]:
train_ds, test_ds = random_split(lazy_dataset, test_size=0.2)
len_train_ds = len(train_ds)
len_test_ds = len(test_ds)
len_lazy_dataset = len(lazy_dataset)
print(len_train_ds)
print(len_test_ds)
print(len_test_ds / len_lazy_dataset)
len_train_ds + len_test_ds == len_lazy_dataset

In [ ]:
embedding_dim = 20
embedding_dims = dict.fromkeys(
    lazy_dataset.get_columns_by_type(column_type=ColumnType.CAT), embedding_dim
)
embedding_dims["vc"] = 10
embedding_params = lazy_dataset.cat_embedding_params(embedding_dims)